# Clasificación Predictiva – Salud de Baterías

En esta sección aplicaremos **algoritmos de Machine Learning (Clasificación)** utilizando los datos procesados en la etapa de Minería de Datos.

El objetivo es predecir si el sistema entrará en un estado de **ALERTA_BATERIA** (riesgo de descarga profunda) analizando únicamente las variables operativas (temperatura, potencia de paneles, energía generada).

## Importar las librerías necesarias

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, roc_auc_score

import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas.")

## Carga de datos y selección de variables predictoras

In [ ]:
# Cargar el dataset diario desde la carpeta de minería
df = pd.read_csv("../data/mineria/dataset_mineria_diario.csv")

# Variables predictoras (Features) y la variable objetivo (Target)
# Omitimos variables que incluyan directamente el voltaje/soc de la batería para que el modelo no haga trampa.
columnas_pred = ['pv_power_W_MEAN', 'controller_temp_C_MEAN', 'energy_generated_today_kWh_MAX']
columnas_existentes = [c for c in columnas_pred if c in df.columns]

X = df[columnas_existentes]
y = df['ALERTA_BATERIA']

print("Variables predictoras (X):", list(X.columns))
print("Variable a predecir (y): ALERTA_BATERIA")
print("Balance de clases:\n", y.value_counts(normalize=True).round(2))

## División en conjuntos de entrenamiento y prueba (Train/Test Split)

Separamos el 70% de los días para enseñar al modelo y el 30% restante para evaluar qué tan bien aprendió a generalizar.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Dimensiones del conjunto de entrenamiento:", X_train.shape)
print("Dimensiones del conjunto de prueba:", X_test.shape)

## Estandarización de variables
Los modelos como KNN y Regresión Logística son sensibles a la magnitud numérica (ej. Potencia está en los miles, Temperatura en decenas). Debemos escalar todo a la misma proporción.

In [ ]:
# Crear y ajustar el escalador a los datos de entrenamiento
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Variables estandarizadas con éxito.")

## Entrenamiento de modelos de clasificación

In [ ]:
print("=== Entrenamiento de Modelos ===\n")

# 1. Árbol de Decisión
model_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
model_tree.fit(X_train, y_train)  # No requiere variables escaladas
print("Árbol de Decisión entrenado.")

# 2. K-Nearest Neighbors (KNN)
model_knn = KNeighborsClassifier(n_neighbors=5)
model_knn.fit(X_train_scaled, y_train)
print("KNN entrenado.")

# 3. Regresión Logística
model_log = LogisticRegression(random_state=42)
model_log.fit(X_train_scaled, y_train)
print("Regresión Logística entrenada.")

## Evaluación de modelos
Definimos una función para calcular **Accuracy, Precision, Recall y F1-score**, y la matriz de confusión.

In [ ]:
def evaluar_modelo(nombre, modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    
    # Cálculo de métricas
    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)
    
    print(f"--- {nombre} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}\n")
    
    return [acc, prec, rec, f1]

resultados = []

# Evaluar usando test set (cuidado: el árbol usa X_test sin escalar, los otros usan X_test_scaled)
resultados.append(evaluar_modelo("Árbol de Decisión", model_tree, X_test, y_test))
resultados.append(evaluar_modelo("KNN (Vecinos Cercanos)", model_knn, X_test_scaled, y_test))
resultados.append(evaluar_modelo("Regresión Logística", model_log, X_test_scaled, y_test))

## Comparativa Visual de Resultados

In [ ]:
df_resultados = pd.DataFrame(resultados, columns=["Accuracy","Precision","Recall","F1-Score"],
                             index=["Árbol de Decisión", "KNN", "Regresión Logística"])

df_resultados.plot(kind='bar', figsize=(10,6), colormap='viridis')
plt.title('Comparativa de Métricas por Modelo')
plt.ylabel('Puntuación (0 a 1)')
plt.xticks(rotation=0)
plt.legend(loc='lower center')
plt.tight_layout()
plt.show()

## Curva ROC y AUC
Mide la capacidad del modelo para distinguir el "Día Seguro" vs el "Día con Alerta". Entre más cerca a 1.0 esté el AUC, mejor.

In [ ]:
y_proba_log = model_log.predict_proba(X_test_scaled)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba_log)
auc = roc_auc_score(y_test, y_proba_log)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Receiver Operating Characteristic - Regresión Logística')
plt.legend(loc="lower right")
plt.show()

## Importancia de Variables (Feature Importance)
Extraemos del Árbol de Decisión cuáles variables fueron las determinantes para lanzar una Alerta.

In [ ]:
importancia = pd.Series(model_tree.feature_importances_, index=X_train.columns)
importancia.sort_values().plot(kind='barh', figsize=(6,4), color='skyblue')
plt.title('Importancia de las variables (Árbol de Decisión)')
plt.xlabel('Peso en la decisión')
plt.show()

## Matriz de Confusión Normalizada
La matriz muestra gráficamente dónde se equivocó el modelo. Nos interesa especialmente los **Falsos Negativos** (Dijo que no habría alerta, pero la batería sí cayó).

In [ ]:
cm = confusion_matrix(y_test, model_log.predict(X_test_scaled), normalize='true')
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=['Pred. Normal', 'Pred. Alerta'], 
            yticklabels=['Real Normal', 'Real Alerta'])
plt.title('Matriz de Confusión - Regresión Logística')
plt.ylabel('Valor Real')
plt.xlabel('Valor Predicho')
plt.show()

# Exportar Predicciones
Guardamos un dataset para contrastar lo que pasó realmente vs lo que el modelo intuyó.

In [ ]:
df_predicciones = df.loc[y_test.index, ['station_name', 'FECHA', 'ALERTA_BATERIA']].copy()
df_predicciones['Prediccion_Modelo_Logistico'] = model_log.predict(X_test_scaled)
df_predicciones['Probabilidad_Alerta_%'] = (model_log.predict_proba(X_test_scaled)[:,1] * 100).round(2)

df_predicciones.to_csv("../data/mineria/predicciones_alertas_modelo.csv", index=False)
print("Predicciones exportadas correctamente a 'predicciones_alertas_modelo.csv'.")
display(df_predicciones.head())

# Conclusión

En esta etapa hemos culminado el ciclo de análisis predictivo. Alimentamos varios algoritmos de Machine Learning con el historial de nuestros controladores solares y comprobamos qué modelo previene mejor una **Alerta de Batería**. 

Con estas métricas (Recall, ROC y Matrices de Confusión), tienes evidencia matemática contundente de la fiabilidad del proyecto de Big Data aplicado a sistemas de energía renovable off-grid.